In [ ]:
from .config import PathConfig #Path config for dynamic object co-ordination

In [ ]:
PathConfig('emotion', 'data')
PROJECT_DIR = paths.project
DATA_DIR = paths.data
BASE_DIR = paths.root

In [ ]:
import torch
from torchvision import transforms, datasets
from torch.utils.data import WeightedRandomSampler

from .train import trainModel
from .model import createModel
from .utils import EarlyStopping, unfreezeLayers, checkDataLoading, evalModel

In [5]:
""" Run this if using drive 
# Copy once from Drive
!cp /content/drive/MyDrive/emotrain/emo_dataset.tar.gz /content/

# Extract locally
!mkdir /content/data/
!tar -xzf /content/emo_dataset.tar.gz -C /content/data/

""" 

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
print("🚀 Starting emotion recognition training...")

# Check CUDA
if torch.cuda.is_available():
    print(f"✅ CUDA available: {torch.cuda.get_device_name()}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory // 1024**2} MB")
else:
    print("⚠️ CUDA not available, using CPU")

base = self.root

# Simpler transforms to start with
train_transform = transforms.Compose([
    transforms.Resize((64, 64), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.0, hue=0.0),
    transforms.RandomHorizontalFlip(0.2),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.12), ratio=(0.3, 3.3), value='random'),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

val_transform = transforms.Compose([
    transforms.Resize((64, 64), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

print("📁 Loading datasets...")
try:
    train_dataset = datasets.ImageFolder(f"{base}/data/raf-db1/train", transform=train_transform)
    val_dataset = datasets.ImageFolder(f"{base}/data/raf-db1/val", transform=val_transform)
    test_dataset = datasets.ImageFolder(f"{base}/data/raf-db1/test", transform=val_transform)

    print("✅ Datasets loaded successfully")
except Exception as e:
    print(f"❌ Dataset loading error: {e}")


# Create a sampler for class imbalances
y_train = [train_dataset.targets[i] for i in range(len(train_dataset))]

# Get the count of each class as an array
class_sample_count = np.array(
    [len(np.where(y_train == t)[0]) for t in np.unique(y_train)]
)

# Find the weights for each class
weight = 1. / class_sample_count
samples_weight = np.array([weight[t] for t in y_train])
samples_weight = torch.from_numpy(samples_weight)

# Create the sampler
sampler = WeightedRandomSampler(samples_weight.type('torch.DoubleTensor'), len(samples_weight))

# Use smaller batch size for debugging
batch_size = 64
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=batch_size,
    sampler=sampler,
    num_workers=2,
    pin_memory=True if torch.cuda.is_available() else False
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True if torch.cuda.is_available() else False
)

val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True if torch.cuda.is_available() else False
)

class_names = list(train_dataset.class_to_idx.keys())
print("Classes:", class_names)
print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

print(f"Batch size: {batch_size}")

# Test data loading first
if not checkDataLoading(train_loader, val_loader):
    print("Data loading test failed!")


# Run the full pipeline
history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

final_model = trainModel(train_loader, val_loader, history)
# You can choose to save history post training to a json file

In [ ]:
evalModel(final_model, test_loader)